In [4]:
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import os

In [5]:

# Prüfe GPU-Nutzung
#if torch.cuda.is_available():
#    device = 'cuda'
#    print("GPU-Modus: CUDA")
#else:
device = 'cpu'
print("ACHTUNG: Läuft auf CPU! GPU nicht gefunden.")

# Modell laden (GPU, falls verfügbar)
model = YOLO('yolo11n-pose.pt')
model.to(device)

# Videoquelle (Pfad anpassen, oder 0 für Webcam)
video_path = r'C:\Users\mtwar\Documents\MAS Data Science\CAS Machine Intelligence\Deep Learning\LNW\5_Video\x_archive\chairlift_vid1.mp4'
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print(f"Video konnte nicht geöffnet werden: {video_path}")
    exit()


ACHTUNG: Läuft auf CPU! GPU nicht gefunden.


In [6]:

# Video-Stream verarbeiten
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Pose-Detektion auf GPU/CPU
    results = model(frame, device=device)

    # Pose-Overlay ins Bild zeichnen
    out_frame = results[0].plot()

    # Im separaten Fenster anzeigen (ESC zum Beenden)
    cv2.imshow("YOLO11n Pose Detection (Live)", out_frame)
    if cv2.waitKey(1) & 0xFF == 27:  # ESC
        break

cap.release()
cv2.destroyAllWindows()



0: 480x640 (no detections), 292.7ms
Speed: 9.8ms preprocess, 292.7ms inference, 7.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 227.8ms
Speed: 8.3ms preprocess, 227.8ms inference, 3.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 265.5ms
Speed: 9.6ms preprocess, 265.5ms inference, 14.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 316.2ms
Speed: 9.1ms preprocess, 316.2ms inference, 6.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 272.2ms
Speed: 9.2ms preprocess, 272.2ms inference, 5.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 257.7ms
Speed: 8.1ms preprocess, 257.7ms inference, 4.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 314.1ms
Speed: 10.2ms preprocess, 314.1ms inference, 6.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 3 persons, 239.9ms
Speed: 8.2ms preprocess, 239.9ms inference, 6.1ms postp

KeyboardInterrupt: 

In [7]:
import cv2
import numpy as np

# Laden des OpenPose-Modells
net = cv2.dnn.readNetFromCaffe("pose_deploy.prototxt", "pose_iter_584000.caffemodel")

# Laden des Videos
video_path = r'C:\Users\mtwar\Documents\MAS Data Science\CAS Machine Intelligence\Deep Learning\LNW\5_Video\x_archive\chairlift_vid1.mp4'
cap = cv2.VideoCapture(video_path)

# Variablen für das Tracking
positions = []
speeds = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Vorbereitung des Frames für das Modell
    frame_width = frame.shape[1]
    frame_height = frame.shape[0]
    inp_blob = cv2.dnn.blobFromImage(frame, 1.0 / 255, (368, 368), (0, 0, 0), swapRB=False, crop=False)
    net.setInput(inp_blob)
    output = net.forward()

    # Extraktion der Positionsdaten
    points = []
    for i in range(15):  # OpenPose COCO Körperpunkte
        prob_map = output[0, i, :, :]
        min_val, prob, min_loc, point = cv2.minMaxLoc(prob_map)
        x = (frame_width * point[0]) / output.shape[3]
        y = (frame_height * point[1]) / output.shape[2]
        if prob > 0.1:  # Schwellenwert für die Erkennung
            points.append((int(x), int(y)))
        else:
            points.append(None)

    # Berechnung des Mittelpunkts der erkannten Punkte
    center_point = None
    valid_points = [p for p in points if p is not None]
    if len(valid_points) > 0:
        center_point = (int(np.mean([p[0] for p in valid_points])), int(np.mean([p[1] for p in valid_points])))
        positions.append(center_point)

    # Berechnung der Geschwindigkeit
    if len(positions) > 1:
        dx = positions[-1][0] - positions[-2][0]
        dy = positions[-1][1] - positions[-2][1]
        distance = np.sqrt(dx**2 + dy**2)
        speed = distance
        speeds.append(speed)
    else:
        speed = 0

    # Zeichnen der Punkte und des Mittelpunkts
    for i, p in enumerate(points):
        if p:
            cv2.circle(frame, p, 5, (0, 255, 255), thickness=-1, lineType=cv2.FILLED)
            cv2.putText(frame, "{}".format(i), p, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, lineType=cv2.LINE_AA)

    if center_point:
        cv2.circle(frame, center_point, 8, (0, 0, 255), thickness=-1, lineType=cv2.FILLED)
        # Anzeige der Position und Geschwindigkeit
        cv2.putText(frame, f"Position: {center_point}", (center_point[0] - 100, center_point[1] - 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
        cv2.putText(frame, f"Speed: {speed:.2f} px/frame", (center_point[0] - 100, center_point[1] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

    cv2.imshow("Frame", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

error: OpenCV(4.11.0) D:\a\opencv-python\opencv-python\opencv\modules\dnn\src\caffe\caffe_io.cpp:1126: error: (-2:Unspecified error) FAILED: fs.is_open(). Can't open "pose_deploy.prototxt" in function 'cv::dnn::ReadProtoFromTextFile'


In [ ]:
import numpy as np
import torch
import seaborn
from collections import defaultdict

# Laden des YOLO-Modells
model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)  # Beispiel mit YOLOv5

# Laden des Videos
#video_path = r'C:\Users\mtwar\Documents\MAS Data Science\CAS Machine Intelligence\Deep Learning\LNW\5_Video\WhatsApp Video 2025-05-12 at 11.45.15.mp4'
#cap = cv2.VideoCapture(video_path)

# Setze die Startposition auf 15 Sekunden (15000 Millisekunden)
#start_time_ms = 15000
#cap.set(cv2.CAP_PROP_POS_MSEC, start_time_ms)

#video_path = r'C:\Users\mtwar\Documents\MAS Data Science\CAS Machine Intelligence\Deep Learning\LNW\5_Video\chairlift_vid1.mp4'
video_path = r'C:\Users\mtwar\Documents\MAS Data Science\CAS Machine Intelligence\Deep Learning\LNW\5_Video\Normal_short.mp4'
cap = cv2.VideoCapture(video_path)

# Variablen für das Tracking
person_positions = {}  # Dictionary zur Speicherung der Positionen jeder Person
person_position_history = defaultdict(lambda: [])  # Speichert die Positionshistorie für jede Person
next_person_id = 0

# Schwellenwert für die Box-Größe und Konfidenz
MIN_BOX_AREA = 800 
CONF_PERSON = 0.6

def calculate_iou(box1, box2):
    """Berechnet die Intersection over Union (IoU) zwischen zwei Boxen."""
    x1, y1, x2, y2 = box1
    x1_p, y1_p, x2_p, y2_p = box2

    xx1 = max(x1, x1_p)
    yy1 = max(y1, y1_p)
    xx2 = min(x2, x2_p)
    yy2 = min(y2, y2_p)

    intersection_area = max(0, xx2 - xx1) * max(0, yy2 - yy1)

    box1_area = (x2 - x1) * (y2 - y1)
    box2_area = (x2_p - x1_p) * (y2_p - y1_p)

    union_area = box1_area + box2_area - intersection_area

    iou = intersection_area / union_area if union_area > 0 else 0
    return iou

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)

    current_frame_boxes = []
    for *xyxy, conf, cls in results.xyxy[0]:
        if int(cls) == 0 and conf > CONF_PERSON:  # Klasse 0 ist Person in COCO
            x1, y1, x2, y2 = map(int, xyxy)
            box_width = x2 - x1
            box_height = y2 - y1
            box_area = box_width * box_height

            if box_area > MIN_BOX_AREA:
                current_frame_boxes.append((x1, y1, x2, y2))

    assigned_ids = set()
    new_person_positions = {}

    for box in current_frame_boxes:
        max_iou = 0
        best_person_id = None

        for person_id, prev_box in person_positions.items():
            iou = calculate_iou(box, prev_box)
            if iou > max_iou and iou > 0.5:
                max_iou = iou
                best_person_id = person_id

        if best_person_id is not None:
            assigned_ids.add(best_person_id)
            new_person_positions[best_person_id] = box
        else:
            new_person_id = next_person_id
            new_person_positions[new_person_id] = box
            next_person_id += 1

    for person_id, box in new_person_positions.items():
        x1, y1, x2, y2 = box
        center_point = ((x1 + x2) // 2, (y1 + y2) // 2)

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.circle(frame, center_point, 5, (0, 0, 255), -1)

        # Aktualisierung der Positionshistorie
        person_position_history[person_id].append(center_point)
        if len(person_position_history[person_id]) > 5:
            person_position_history[person_id].pop(0)

        # Berechnung der Geschwindigkeit als Mittelwert über die letzten 5 Frames
        if len(person_position_history[person_id]) > 1:
            distances = []
            for i in range(1, len(person_position_history[person_id])):
                prev_center = person_position_history[person_id][i-1]
                current_center = person_position_history[person_id][i]
                dx = current_center[0] - prev_center[0]
                dy = current_center[1] - prev_center[1]
                distance = np.sqrt(dx**2 + dy**2)
                distances.append(distance)

            avg_speed = np.mean(distances)
        else:
            avg_speed = 0

        if person_id % 2 == 0:
            cv2.putText(frame, f"ID: {person_id}", (x1, y1 - 50), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
            cv2.putText(frame, f"Position: {center_point}", (x1, y1 - 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
            cv2.putText(frame, f"Avg Speed: {avg_speed:.2f} px/frame", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
        else:
            cv2.putText(frame, f"ID: {person_id}", (x1, y2 + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
            cv2.putText(frame, f"Position: {center_point}", (x1, y2 + 40), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
            cv2.putText(frame, f"Avg Speed: {avg_speed:.2f} px/frame", (x1, y2 + 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

    person_positions = new_person_positions

    cv2.imshow("Frame", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Using cache found in C:\Users\mtwar/.cache\torch\hub\ultralytics_yolov5_master
C:\Users\mtwar/.cache\torch\hub\ultralytics_yolov5_master\utils\general.py:32: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources as pkg


requirements: Ultralytics requirement ['gitpython>=3.1.30'] not found, attempting AutoUpdate...

requirements: AutoUpdate success  0.8s
WARNING requirements: Restart runtime or rerun command for updates to take effect



YOLOv5  2025-6-2 Python-3.12.10 torch-2.7.0+cpu CPU

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 
C:\Users\mtwar/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\mtwar/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\mtwar/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\mtwar/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is